# DistilBART Pipeline: 
Input text → Encoder → Decoder → Generated output.  

# Example:
Article -> BART -> Summary  

# Explaination;

DistilBART is a smaller version or little brother from a BART model. The distilBART is faster in speed, but smaller in size, this decreases the performance a small bit.  
The decoder and encoder differences are:
-   BART = 12 decoder, 12 encoder, 400M paramters.
-   DistilBART 6 decoder, 6 encoder, 200M parameters.  

# Work:
Input text  
   ↓  
DistilBART (Transformers + PyTorch)  
   ↓  
Summary / key content  
   ↓  
spaCy  
   ↓  
Entities: contributors, organizations, dates, etc.

The current metadata extraction works best on semi-structured project texts. When fields such as dates, contact persons, or researcher roles are only implied and not explicitly labeled, rule-based extraction becomes less reliable. In those cases, additional contextual heuristics or model-based extraction are needed.

Pip install the necesarry libraries

In [1]:
# Libaries you need
# %pip install transformers
# %pip install torch
# %pip install spacy
# %python -m spacy download en_core_web_sm

Import all the necesarry libaries

In [2]:
# install libraries
import pandas as pd
import os
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import json
import time
import re
import spacy

c:\Users\joeri\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load the data from silver_nlp layer

In [3]:
# Import text data from silver layer
df = pd.read_json("../Data/silver/doc_01_silver_nlp.json")

print(df.head())

  document_id                                       cleaned_text  \
0      doc_01  Delta Water Innovation Hub\n\nProject ID: DWIH...   

                                              tokens  \
0  [delta, water, innovation, hub, project, id, s...   

                                           sentences  \
0  [Delta Water Innovation Hub\n\nProject ID: DWI...   

                                            entities  
0  [[Delta Water Innovation Hub\n\nProject ID, OR...  


Here we load the distilBART model, this could take a while the first time since its 1GB load.

In [4]:
DistBART = "sshleifer/distilbart-cnn-12-6"

tokenizer = AutoTokenizer.from_pretrained(DistBART)
model = AutoModelForSeq2SeqLM.from_pretrained(DistBART)

Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 358/358 [00:00<00:00, 48531.38it/s]
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Here we go over the model with a summarization.

In [5]:
def summarize_distilbart(text,
                         max_input_len=1024,
                         max_output_len=200,
                         min_output_len=40):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len
    )

    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_output_len,
            min_length=min_output_len,
            num_beams=4,
            early_stopping=True
        )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    return summary

In [6]:
text = df.loc[0, "cleaned_text"]

summary = summarize_distilbart(text)

print(summary)

 Delta Water Innovation Hub project focuses on the reuse of treated water streams for agriculture and industrial applications in Zeeland . The project investigates how sensor systems, predictive models and decision-support dashboards can support sustainable water management in coastal regions .


In [ ]:
# Build the gold output structure
gold_output = {
    "document_id": df.loc[0, "document_id"],
    "model": "distilBART",
    "raw_text": text,
    "summary": summary
}

# Save to gold layer
with open("../Data/gold/distilBART/distilBART.json", "w") as f:
    json.dump(gold_output, f, indent=4)

print("Saved DistilBART summary to gold layer.")

Saved DistilBART summary to gold layer.


In [8]:
def simple_tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

def evaluate_summary(original_text, summary_text, source_entities=None):
    original_tokens = simple_tokenize(original_text)
    summary_tokens = simple_tokenize(summary_text)

    original_len = len(original_tokens)
    summary_len = len(summary_tokens)

    compression_ratio = summary_len / original_len if original_len > 0 else 0

    # entity preservation
    preserved_entities = []
    missed_entities = []

    source_entities = source_entities or []
    summary_lower = summary_text.lower()

    for ent in source_entities:
        ent_text = ent[0] if isinstance(ent, (list, tuple)) and len(ent) >= 1 else str(ent)
        ent_text_clean = " ".join(ent_text.split()).strip()

        if ent_text_clean and ent_text_clean.lower() in summary_lower:
            preserved_entities.append(ent_text_clean)
        else:
            missed_entities.append(ent_text_clean)

    total_entities = len(source_entities)
    entity_preservation_score = (
        len(preserved_entities) / total_entities if total_entities > 0 else None
    )

    return {
        "original_token_count": original_len,
        "summary_token_count": summary_len,
        "compression_ratio": compression_ratio,
        "preserved_entities": preserved_entities,
        "missed_entities": missed_entities,
        "entity_preservation_score": entity_preservation_score
    }

In [9]:
start = time.time()
summary = summarize_distilbart(df.loc[0, "cleaned_text"])
runtime_seconds = time.time() - start

eval_result = evaluate_summary(
    original_text=df.loc[0, "cleaned_text"],
    summary_text=summary,
    source_entities=df.loc[0, "entities"]
)

eval_result["runtime_seconds"] = runtime_seconds

print(summary)
print(eval_result)

 Delta Water Innovation Hub project focuses on the reuse of treated water streams for agriculture and industrial applications in Zeeland . The project investigates how sensor systems, predictive models and decision-support dashboards can support sustainable water management in coastal regions .
{'original_token_count': 133, 'summary_token_count': 40, 'compression_ratio': 0.3007518796992481, 'preserved_entities': ['Zeeland'], 'missed_entities': ['Delta Water Innovation Hub Project ID', '2026-02-01', '2028-12-31', 'Lisa Vermeer', 'HZ Kenniscentrum Zeeuwse Samenleving and Water Technology', 'Lisa Vermeer', 'Tom de Bruin', 'Netherlands'], 'entity_preservation_score': 0.1111111111111111, 'runtime_seconds': 1.9772193431854248}


In [10]:
def sentence_coverage_score(source_sentences, summary_text, threshold=0.2):
    summary_words = set(simple_tokenize(summary_text))
    covered = 0

    for sent in source_sentences:
        sent_words = set(simple_tokenize(sent))
        if not sent_words:
            continue

        overlap = len(summary_words & sent_words) / len(sent_words)
        if overlap >= threshold:
            covered += 1

    return covered / len(source_sentences) if source_sentences else 0

In [11]:
coverage = sentence_coverage_score(df.loc[0, "sentences"], summary)

eval_result["sentence_coverage_score"] = coverage
print(eval_result)

{'original_token_count': 133, 'summary_token_count': 40, 'compression_ratio': 0.3007518796992481, 'preserved_entities': ['Zeeland'], 'missed_entities': ['Delta Water Innovation Hub Project ID', '2026-02-01', '2028-12-31', 'Lisa Vermeer', 'HZ Kenniscentrum Zeeuwse Samenleving and Water Technology', 'Lisa Vermeer', 'Tom de Bruin', 'Netherlands'], 'entity_preservation_score': 0.1111111111111111, 'runtime_seconds': 1.9772193431854248, 'sentence_coverage_score': 1.0}


In [ ]:
evaluation_row = {
    "document_id": df.loc[0, "document_id"],
    "model": "distilBART",
    "runtime_seconds": runtime_seconds,
    "original_token_count": eval_result["original_token_count"],
    "summary_token_count": eval_result["summary_token_count"],
    "compression_ratio": eval_result["compression_ratio"],
    "entity_preservation_score": eval_result["entity_preservation_score"],
    "sentence_coverage_score": eval_result["sentence_coverage_score"]
}

print(evaluation_row)

gold_eval_output = {
    "document_id": df.loc[0, "document_id"],
    "model": "distilBART",
    "evaluation": evaluation_row,
    
}

# Save to gold layer
with open("../Data/gold/distilBART/distilBART_evaluation.json", "w") as f:
    json.dump(gold_eval_output, f, indent=4)

print("Saved DistilBART evaluation to gold layer.")

{'document_id': 'doc_01', 'model': 'distilBART', 'runtime_seconds': 1.9772193431854248, 'original_token_count': 133, 'summary_token_count': 40, 'compression_ratio': 0.3007518796992481, 'entity_preservation_score': 0.1111111111111111, 'sentence_coverage_score': 1.0}
Saved DistilBART evaluation to gold layer.


What do these numbers mean?

    "document_id": "doc_01",
    "model": "distilBART",
    "evaluation": {
        "document_id": "doc_01",
        "model": "distilBART",
        "runtime_seconds": 1.470646619796753,
        "original_token_count": 317,
        "summary_token_count": 35,
        "compression_ratio": 0.11041009463722397,
        "entity_preservation_score": 0.4,
        "sentence_coverage_score": 0.6190476190476191


document_id
Identifier of the processed document. Used to track which text the summary was generated from.

model
Name of the model that generated the summary (e.g., DistilBART, BART, mBART, BERT).

runtime_seconds
Time the model needed to generate the summary. Used to compare model efficiency.

original_token_count
Number of tokens (words) in the original document.

summary_token_count
Number of tokens in the generated summary.

compression_ratio
How much the text was shortened.

entity_preservation_score
Measures how many named entities from the original text appear in the summary.

# Now we go on with NER, to gather some key information out of the text

In [ ]:
nlp = spacy.load("en_core_web_sm")

# TO-DO: implement with API and retrieve actual research groups from KMP
RESEARCH_GROUPS = [
    "Aquaculture in Delta Areas",
    "Assetmanagement",
    "Biobased Bouwen",
    "Building with Nature",
    "Data Science",
    "Delta Power",
    "Excellence and Innovation in Education",
    "HZ Kenniscentrum Kusttoerisme",
    "HZ Kenniscentrum Ondernemen en Innoveren",
    "HZ Kenniscentrum Zeeuwse Samenleving",
    "HZ Nexus",
    "Healthy Region",
    "Kunst Cultuur Transitie",
    "Marine Biobased Chemie",
    "Ouderenzorg",
    "Resilient Deltas",
    "Supply Chain Innovation",
    "Water Technology"
]

In [14]:
def extract_people(text):
    doc = nlp(text)
    people = [ent.text.strip() for ent in doc.ents if ent.label_ == "PERSON"]
    return sorted(set(people))

In [15]:
def extract_dates(text):
    iso_dates = re.findall(r"\b\d{4}-\d{2}-\d{2}\b", text)
    return sorted(set(iso_dates))

In [16]:
def extract_title(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[0] if lines else ""

In [17]:
def extract_description(summary, fallback_text):
    if summary and summary.strip():
        return summary.strip()
    return fallback_text[:500].strip()

In [18]:
def extract_research_groups(text, research_groups):
    text_lower = text.lower()
    matches = []

    for group in research_groups:
        if group.lower() in text_lower:
            matches.append(group)

    return sorted(set(matches))

In [19]:
def extract_project_metadata(text, summary=None, document_id=None):
    people = extract_people(text)
    dates = extract_dates(text)
    research_groups = extract_research_groups(text, RESEARCH_GROUPS)

    metadata = {
        "id": document_id or "",
        "title": extract_title(text),
        "description": extract_description(summary, text),
        "contact_person": people[0] if people else "",
        "start_date": dates[0] if len(dates) > 0 else "",
        "end_date": dates[1] if len(dates) > 1 else "",
        "research_groups": research_groups,
        "researchers": people
    }

    return metadata

In [20]:
text = df.loc[0, "cleaned_text"]
document_id = df.loc[0, "document_id"]

metadata = extract_project_metadata(
    text=text,
    summary=summary,
    document_id=document_id
)

print(metadata)

{'id': 'doc_01', 'title': 'Delta Water Innovation Hub', 'description': 'Delta Water Innovation Hub project focuses on the reuse of treated water streams for agriculture and industrial applications in Zeeland . The project investigates how sensor systems, predictive models and decision-support dashboards can support sustainable water management in coastal regions .', 'contact_person': 'Lisa Vermeer', 'start_date': '2026-02-01', 'end_date': '2028-12-31', 'research_groups': ['HZ Kenniscentrum Zeeuwse Samenleving', 'Water Technology'], 'researchers': ['Lisa Vermeer', 'Tom de Bruin']}


In [ ]:
combined_output = {
    "document_id": df.loc[0, "document_id"],
    "model": "distilBART",

    "summary": summary,

    "metadata": metadata,

    "evaluation": {
        "runtime_seconds": runtime_seconds,
        "original_token_count": eval_result["original_token_count"],
        "summary_token_count": eval_result["summary_token_count"],
        "compression_ratio": eval_result["compression_ratio"],
        "entity_preservation_score": eval_result["entity_preservation_score"],
        "sentence_coverage_score": eval_result["sentence_coverage_score"]
    }
}

with open("../Data/gold/distilBART/distilBART_output.json", "w") as f:
    json.dump(combined_output, f, indent=4)

print("Saved combined DistilBART output to gold layer.")

Saved combined DistilBART output to gold layer.
